# Phase 0 — Headroom pre-check**What this does:** measures how well Qwen3.5-4B answers 90 two-hop reasoning questions, with nothing modified. No lens, no ablation.**Why:** if the model can't answer these well to begin with, there's nothing for a later ablation to destroy — and Control A couldn't tell "broken instrument" apart from "model was never able to do the task."**Before you run anything:** write down and commit your pass threshold. See `THRESHOLD_headroom.md`. A threshold picked after seeing the number isn't a threshold.---### Turn the GPU on first`Runtime` → `Change runtime type` → Hardware accelerator → **T4 GPU** → Save.This is the single most common thing to forget. Cell 1 checks it for you.

## Cell 1 — Confirm the GPU is on

In [ ]:
import torch, subprocessprint("GPU visible to PyTorch:", torch.cuda.is_available())if torch.cuda.is_available():    print("Device:", torch.cuda.get_device_name(0))    print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())else:    print("\n>>> STOP. No GPU. Runtime > Change runtime type > T4 GPU > Save, then re-run this cell.")

## Cell 2 — Install librariesTakes about a minute. A pip dependency warning in red is normal and safe to ignore.

In [ ]:
!pip -q install -U transformers accelerateimport transformers; print("transformers", transformers.__version__)

## Cell 3 — Get the question setClones Anthropic's public repo. We only need one file from it: the 90 two-hop prompts.

In [ ]:
!git clone -q --depth 1 https://github.com/anthropics/jacobian-lens.gitimport jsonitems = json.load(open("jacobian-lens/data/experiments/probe-swap.json"))["items"]print(f"Loaded {len(items)} prompts\n")for it in items[:2]:    print("prompt    :", repr(it["prompt"]))    print("answer    :", repr(it["answer"]))    print("(the model has to work out the intermediate step:", it["intermediate"], ")\n")

## Cell 4 — Write the checking scriptThis writes `headroom_check.py` into the Colab session. Nothing to upload — the whole script is in this cell.Two things it handles that would otherwise silently ruin the result:- **Trailing spaces.** 29 of the 90 prompts end with a space and 61 don't, so the correct continuation is `" Portuguese"` for some and `"Portuguese"` for others. Getting this wrong scores a large chunk of correct answers as wrong.- **Multi-token answers.** Words like "California" and "Budapest" are several tokens. Strict single-next-token matching marks those wrong even when the model is right.

In [ ]:
%%writefile headroom_check.py"""Phase 0 headroom pre-check — DECISION_control_A.md §4.6.Measures unablated accuracy on the 90 two-hop prompts of probe-swap.json.No lens. No harness. No ablation. One forward pass per prompt.This decides the model. Per DECISION_control_A.md §4.6, the paper's ablationresult depends on the unablated model being near ceiling; if accuracy is lowthere is nothing for ablation to remove, and a failed Control A would be causedby model selection rather than by a broken instrument — which is precisely theattribution Control A exists to make possible.**Write and commit your threshold before running this.** See the docstring of`main()`. A threshold chosen after seeing the number is not a threshold.Usage:    python headroom_check.py --model Qwen/Qwen3.5-4B \        --data data/experiments/probe-swap.json --out results/raw/headroom/"""from __future__ import annotationsimport argparseimport jsonimport subprocessfrom collections import Counter, defaultdictfrom pathlib import Pathimport torchfrom transformers import AutoModelForCausalLM, AutoTokenizer# --- scoring ---------------------------------------------------------------def normalise(prompt: str, answer: str) -> tuple[str, str]:    """Fix the trailing-whitespace inconsistency in probe-swap.json.    29 of 90 prompts end with a space and 61 do not, so the true continuation is    " Portuguese" for some items and "Portuguese" for others. Tokenising the raw    `answer` scores one group wrong regardless of which convention you pick, and    depresses measured accuracy for a reason that has nothing to do with the    model. Normalising both sides removes the artifact.    """    return prompt.rstrip(), " " + answer.strip()@torch.no_grad()def score_item(model, tok, prompt: str, answer: str, device) -> dict:    """Three metrics, because they disagree and the disagreement is informative.    - ``exact``: greedy continuation of len(answer_tokens) equals the answer.      Strict and unambiguous. Recommended primary.    - ``first_token``: greedy next token equals the answer's first token. This      is what "greedy next-token accuracy" most plausibly means, and it is      lenient — it credits a correct first token followed by a wrong completion.    - ``n_answer_tokens``: 1 means the two metrics coincide for this item.    """    prompt, answer = normalise(prompt, answer)    ans_ids = tok(answer, add_special_tokens=False).input_ids    ids = tok(prompt, return_tensors="pt").input_ids.to(device)    out = model.generate(        ids,        max_new_tokens=len(ans_ids),        do_sample=False,        num_beams=1,        pad_token_id=tok.eos_token_id,    )    gen_ids = out[0, ids.shape[1]:].tolist()    return {        "exact": tok.decode(gen_ids).strip().lower() == answer.strip().lower(),        "first_token": bool(gen_ids) and gen_ids[0] == ans_ids[0],        "n_answer_tokens": len(ans_ids),        "generated": tok.decode(gen_ids),    }def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:    """Wilson 95% interval. At n=90 the interval is roughly +/-8-10pp — wide    enough that a threshold should not be set to two decimal places."""    if n == 0:        return (0.0, 0.0)    p = k / n    d = 1 + z**2 / n    c = (p + z**2 / (2 * n)) / d    h = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / d    return (max(0.0, c - h), min(1.0, c + h))# --- main ------------------------------------------------------------------def main() -> None:    """Run the check and write results.    Before running, commit a file recording:      - the threshold on the primary metric that counts as sufficient headroom      - which metric is primary (recommended: ``exact``)      - what you will do if it fails (DECISION_phase0_model.md §4: step *up*,        not down — below some size Control A stops being informative at all)    """    ap = argparse.ArgumentParser()    ap.add_argument("--model", required=True)    ap.add_argument("--data", required=True, help="path to probe-swap.json")    ap.add_argument("--out", required=True)    ap.add_argument("--dtype", default="bfloat16")    ap.add_argument("--limit", type=int, default=None, help="smoke-test on N items")    args = ap.parse_args()    device = "cuda" if torch.cuda.is_available() else "cpu"    tok = AutoTokenizer.from_pretrained(args.model)    model = AutoModelForCausalLM.from_pretrained(        args.model, dtype=getattr(torch, args.dtype), device_map=device    ).eval()    items = json.load(open(args.data))["items"]    if args.limit:        items = items[: args.limit]    rows = []    for i, it in enumerate(items):        r = score_item(model, tok, it["prompt"], it["answer"], device)        r.update(name=it["name"], category=it["category"], answer=it["answer"])        rows.append(r)        print(f"[{i+1}/{len(items)}] {it['name']:<28} "              f"exact={r['exact']:d} first={r['first_token']:d} "              f"({r['n_answer_tokens']}tok) -> {r['generated']!r}")    n = len(rows)    n_exact = sum(r["exact"] for r in rows)    n_first = sum(r["first_token"] for r in rows)    single = [r for r in rows if r["n_answer_tokens"] == 1]    by_cat = defaultdict(list)    for r in rows:        by_cat[r["category"]].append(r["exact"])    summary = {        "model": args.model,        "n": n,        "exact": {"k": n_exact, "acc": n_exact / n, "wilson95": wilson(n_exact, n)},        "first_token": {"k": n_first, "acc": n_first / n, "wilson95": wilson(n_first, n)},        "single_token_subset": {            "n": len(single),            "acc": (sum(r["exact"] for r in single) / len(single)) if single else None,        },        "answer_token_lengths": dict(Counter(r["n_answer_tokens"] for r in rows)),        # Only categories with n>=4 are worth reading; probe-swap.json has a long        # tail of singleton categories where a "0%" is one item.        "by_category_n_ge_4": {            c: {"n": len(v), "acc": sum(v) / len(v)}            for c, v in sorted(by_cat.items()) if len(v) >= 4        },        "git_commit": subprocess.run(            ["git", "rev-parse", "HEAD"], capture_output=True, text=True        ).stdout.strip() or "UNKNOWN",        "config": vars(args),    }    out = Path(args.out)    out.mkdir(parents=True, exist_ok=True)    (out / "rows.json").write_text(json.dumps(rows, indent=2))    (out / "summary.json").write_text(json.dumps(summary, indent=2))    lo, hi = summary["exact"]["wilson95"]    print("\n" + "=" * 60)    print(f"exact       : {n_exact}/{n} = {n_exact/n:.1%}  (95% CI {lo:.1%}-{hi:.1%})")    print(f"first_token : {n_first}/{n} = {n_first/n:.1%}")    print(f"single-token answers: {len(single)}/{n}")    print("=" * 60)    print("Compare against the threshold you committed BEFORE this run.")if __name__ == "__main__":    main()

## Cell 5 — Smoke test on 5 prompts**Don't skip this.** It downloads the model (~8 GB, a few minutes the first time) and runs 5 prompts so you can eyeball the output before committing to the full run.Note `--dtype float16`. The T4 is an older GPU that doesn't handle bfloat16 well. If you get an L4 or A100, you can use `bfloat16` instead.**What you want to see:** generations that look like real attempted answers. If you see punctuation, blank output, or the model continuing the sentence instead of answering, stop and fix the prompt format — don't run the full 90.

In [ ]:
!python headroom_check.py \    --model Qwen/Qwen3.5-4B \    --data jacobian-lens/data/experiments/probe-swap.json \    --out results/smoke/ \    --dtype float16 \    --limit 5

## Cell 6 — The real run, all 90 promptsThe model is already downloaded now, so this is quick — a couple of minutes.

In [ ]:
!python headroom_check.py \    --model Qwen/Qwen3.5-4B \    --data jacobian-lens/data/experiments/probe-swap.json \    --out results/raw/headroom_qwen3.5-4b/ \    --dtype float16

## Cell 7 — Read the summaryCompare the primary metric against the threshold you committed **before** running.`exact` = the full answer matched. `first_token` = only the first token matched, which is lenient — it gives credit for "Cal" when the answer was "California".

In [ ]:
import jsons = json.load(open("results/raw/headroom_qwen3.5-4b/summary.json"))lo, hi = s["exact"]["wilson95"]print(f"exact       : {s['exact']['k']}/{s['n']} = {s['exact']['acc']:.1%}   95% CI {lo:.1%}-{hi:.1%}")print(f"first_token : {s['first_token']['k']}/{s['n']} = {s['first_token']['acc']:.1%}")print(f"\nanswers that are a single token: {s['single_token_subset']['n']}/{s['n']}")print("answer length in tokens:", s["answer_token_lengths"])print("\nby category (only those with 4+ items — the rest are too small to read):")for c, v in s["by_category_n_ge_4"].items():    print(f"   {c:<20} {v['acc']:>6.0%}  (n={v['n']})")

## Cell 8 — Look at what it got wrongThis matters as much as the score. You're checking whether failures are *reasoning* failures or *scoring* failures.- Plausible-but-different answers (e.g. "Brasilia" where "Brazil" was the routing step) → a scoring problem, and your real headroom is higher than the number says.- Punctuation, empty strings, or the sentence continuing → a prompt-format problem. Fix before drawing any conclusion.

In [ ]:
import jsonrows = json.load(open("results/raw/headroom_qwen3.5-4b/rows.json"))wrong = [r for r in rows if not r["exact"]]print(f"{len(wrong)} incorrect out of {len(rows)}\n")for r in wrong[:25]:    print(f"{r['name']:<26} wanted={r['answer']!r:<16} got={r['generated']!r}")

## Cell 9 — Download the resultsColab sessions vanish and take everything with them. Do this before you close the tab.Save both files into `results/raw/headroom_qwen3.5-4b/` in your repo and commit them.

In [ ]:
from google.colab import filesfiles.download("results/raw/headroom_qwen3.5-4b/summary.json")files.download("results/raw/headroom_qwen3.5-4b/rows.json")

---## Afterwards1. Commit both JSON files to your repo.2. Write the lab log entry. Note that `git_commit` in the summary will say `UNKNOWN` — Colab isn't a git checkout — so record your own repo's commit hash in the log by hand.3. Compare against your committed threshold and make the call.**If headroom is insufficient:** step *up* in model size, not down. A pass from a model that can't reason is worse than an honest partial result.